In [1]:
#!/usr/bin/env Rscript
# ================================================================
# Quick inspection of DHS, ATAC, and Overlap CRE files
# ================================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(data.table)
})

Warning message:
“package ‘GenomicRanges’ was built under R version 4.3.3”
Warning message:
“package ‘BiocGenerics’ was built under R version 4.3.2”
Warning message:
“package ‘S4Vectors’ was built under R version 4.3.3”
Warning message:
“package ‘IRanges’ was built under R version 4.3.3”
Warning message:
“package ‘GenomeInfoDb’ was built under R version 4.3.2”
Warning message:
“package ‘data.table’ was built under R version 4.3.3”


In [5]:
#!/usr/bin/env Rscript
# ===================================================================
# Final CRE Data Integrity Check (DHS / ATAC / Overlap)
# ===================================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(data.table)
})

# ---- 1. Define example file paths ----
dhs_file     <- "../ref/CRE_sites_Selective/DHS_sites_Selective3/Cardiac_filtered.hg19.rds_selective.rds"
atac_file    <- "../ref/CRE_sites_Selective/scATAC_sites_Selective3/Cardiomyocyte_filtered.hg19.rds_selective.rds"
overlap_file <- "../ref/CRE_sites_Overlaps/DHS_ATAC_jaccard_overlaps3/Cardiac-Cardiomyocyte.hg19.consensus.overlaps.rds"

# ---- Helper function ----
print_summary <- function(gr, label) {
  cat("\n============================================================\n")
  cat(sprintf("%s: Class = %s | Length = %d\n", label, class(gr)[1], length(gr)))
  if (length(gr) == 0) return(invisible())
  cat("Metadata columns:\n")
  print(colnames(mcols(gr)))
  cat("\nPreview (first 5 rows):\n")
  df <- as.data.frame(gr)
  show_cols <- c("seqnames", "start", "end")
  mc <- colnames(mcols(gr))
  extra <- head(mc, min(4, length(mc)))
  show_cols <- c(show_cols, extra)
  print(df[seq_len(min(5, nrow(df))), show_cols, drop = FALSE])
}

# ---- 2. Load DHS ----
if (file.exists(dhs_file)) {
  cat("\nLoading DHS file:\n", dhs_file, "\n")
  gr_dhs <- readRDS(dhs_file)
  print_summary(gr_dhs, "DHS file")
} else {
  cat("\n❌ DHS file not found:", dhs_file, "\n")
}

# ---- 3. Load ATAC ----
if (file.exists(atac_file)) {
  cat("\nLoading ATAC file:\n", atac_file, "\n")
  gr_atac <- readRDS(atac_file)
  print_summary(gr_atac, "ATAC file")
} else {
  cat("\n❌ ATAC file not found:", atac_file, "\n")
}

# ---- 4. Load Overlap ----
if (file.exists(overlap_file)) {
  cat("\nLoading Overlap file:\n", overlap_file, "\n")
  overlap_obj <- readRDS(overlap_file)
  cat("Object type:", class(overlap_obj), "\n")

  # handle both GRanges and list
  if (inherits(overlap_obj, "GRanges")) {
    gr_overlap <- overlap_obj
    qc_overlap <- NULL
  } else if (is.list(overlap_obj)) {
    gr_overlap <- overlap_obj$overlap_regions %||% overlap_obj$gr %||% overlap_obj[[1]]
    qc_overlap <- overlap_obj$QC %||% overlap_obj$qc %||% overlap_obj$summary %||% NULL
  } else {
    stop("Unknown overlap object type")
  }

  print_summary(gr_overlap, "Overlap regions")
  if (!is.null(qc_overlap)) {
    cat("\nQC summary (if available):\n")
    print(qc_overlap)
  } else {
    cat("\nNo QC summary detected in overlap object.\n")
  }

  if ("overlap_bp" %in% colnames(mcols(gr_overlap))) {
    cat("\nOverlap metrics preview:\n")
    df <- as.data.frame(mcols(gr_overlap))
    cols <- intersect(c("overlap_bp", "union_bp", "jaccard_index",
                        "summit_delta", "ATAC_summit_hg19", "ATAC_summit_in_overlap"),
                      colnames(df))
    print(head(df[, cols, drop=FALSE]))
  }

} else {
  cat("\n❌ Overlap file not found:", overlap_file, "\n")
}

cat("\n✅ Data integrity check completed.\n")



Loading DHS file:
 ../ref/CRE_sites_Selective/DHS_sites_Selective3/Cardiac_filtered.hg19.rds_selective.rds 

DHS file: Class = GRanges | Length = 23767
Metadata columns:
 [1] "summit_hg19"     "core_start_hg19" "core_end_hg19"   "name"           
 [5] "mean_signal"     "peak.count"      "component"       "position"       
 [9] "score"           "CRE_source"     

Preview (first 5 rows):
  seqnames  start    end summit_hg19 core_start_hg19 core_end_hg19      name
1     chr1 766720 766940      766830          766790        766857  1.103021
2     chr1 767517 767786      767660          767569        767751  1.103026
3     chr1 770998 771052      771050          771050        771050  1.103041
4     chr1 800112 800340      800230          800176        800230  1.103138
5     chr1 800764 800975      800860          800850        800942 1.1031405

Loading ATAC file:
 ../ref/CRE_sites_Selective/scATAC_sites_Selective3/Cardiomyocyte_filtered.hg19.rds_selective.rds 

ATAC file: Class = GRanges 

In [7]:
summary(width(gr_dhs))
summary(width(gr_atac))
summary(width(gr_overlap))


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   41.0   180.0   201.0   218.1   241.0  1481.0 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    401     401     401     401     401     401 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  101.0   185.0   201.0   209.9   226.0   386.0 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.01164 0.02858 0.05745 0.08891 0.11748 1.02685 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    427     946    1907    3255    4036   42562 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.01310 0.06321 0.10697 0.12159 0.15940 0.91016 

In [8]:

summary(gr_dhs$score)
summary(gr_atac$score)
summary(gr_overlap$score)


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.01164 0.02858 0.05745 0.08891 0.11748 1.02685 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    427     946    1907    3255    4036   42562 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.01310 0.06321 0.10697 0.12159 0.15940 0.91016 

## Step 1: DHS file finalizaiton

In [14]:
#!/usr/bin/env Rscript
# ===================================================================
# Batch DHS Finalization (fast version)
# Top 10k → collapse ±100 bp → QC + export .rds/.bed
# ===================================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(IRanges)
  library(rtracklayer)
  library(fs)
  library(tools)
  library(data.table)
})

indir  <- "../ref/CRE_sites_Selective/DHS_sites_Selective3"
outdir <- "../ref/CRE_sites_Final/DHS_filtered_final"
dir_create(outdir)
files  <- list.files(indir, pattern="\\.rds$", full.names=TRUE)
cat(sprintf("Found %d DHS files\n", length(files)))

qc_all <- list()

for (file in files) {
  tissue <- file_path_sans_ext(basename(file))
  message("\n--------------------------------------------")
  message(sprintf("Processing %s ...", tissue))

  gr <- readRDS(file)
  n_input <- length(gr)

  # ---- basic QC ----
  keep <- !is.na(gr$summit_hg19) &
          !is.na(gr$core_start_hg19) &
          !is.na(gr$core_end_hg19) &
          !is.na(gr$score)
  gr <- gr[keep]
  if (!length(gr)) next

  # ---- top 10k ----
  gr <- gr[order(gr$score, decreasing=TRUE)]
  gr_top <- gr[seq_len(min(10000L, length(gr)))]
  message(sprintf("Top %d selected", length(gr_top)))

  # ---- 1 bp summits ----
  gr_1bp <- GRanges(seqnames=seqnames(gr_top),
                    ranges=IRanges(start=gr_top$summit_hg19, width=1),
                    strand="*")
  mcols(gr_1bp) <- mcols(gr_top)[, c(
    "name","mean_signal","peak.count","component","score","CRE_source",
    "core_start_hg19","core_end_hg19","summit_hg19"
  )]

  # ---- collapse redundant (±100 bp) ----
  clusters <- reduce(gr_1bp, min.gapwidth=200, ignore.strand=TRUE)
  hits <- findOverlaps(gr_1bp, clusters, ignore.strand=TRUE)
  idx_by_cluster <- split(queryHits(hits), subjectHits(hits))
  best_idx <- vapply(idx_by_cluster,
                     function(ix) ix[which.max(gr_1bp$score[ix])],
                     integer(1))
  gr_nonred <- gr_1bp[best_idx]
  gr_nonred <- gr_nonred[order(gr_nonred$score, decreasing=TRUE)]

  # ---- QC flag ----
  summit_pos <- start(gr_nonred)
  within_core <- (summit_pos >= gr_nonred$core_start_hg19) &
                 (summit_pos <= gr_nonred$core_end_hg19)
  mcols(gr_nonred)$summit_within_core <- within_core

  # ---- QC summary ----
  qc_all[[tissue]] <- data.frame(
    Tissue = tissue,
    Input_Sites = n_input,
    After_QC = length(gr),
    Top10k_Selected = length(gr_top),
    Final_Nonredundant = length(gr_nonred),
    Percent_Retained = round(100 * length(gr_nonred) / n_input, 2),
    Pct_Summit_Within_Core = round(mean(within_core) * 100, 2),
    Mean_Signal_Final = round(mean(gr_nonred$mean_signal, na.rm=TRUE), 4),
    Median_Score_Final = round(median(gr_nonred$score, na.rm=TRUE), 4)
  )

  # ---- save ----
  out_rds <- file.path(outdir, paste0(tissue, "_DHS_filtered_final_summit.hg19.rds"))
  out_bed <- sub("\\.rds$", ".bed", out_rds)
  saveRDS(gr_nonred, out_rds)
  export.bed(gr_nonred, out_bed)
  message(sprintf("✅ %s saved (%d peaks)", tissue, length(gr_nonred)))
}

# ---- write QC summary ----
qc_df <- rbindlist(qc_all)
fwrite(qc_df, file.path(outdir, "DHS_finalization_QC_summary.csv"))
cat("\n✅ DHS batch finalization complete. QC summary written.\n")


Found 16 DHS files



--------------------------------------------

Processing Cancer_epithelial_filtered.hg19.rds_selective ...

Top 10000 selected

✅ Cancer_epithelial_filtered.hg19.rds_selective saved (9992 peaks)


--------------------------------------------

Processing Cardiac_filtered.hg19.rds_selective ...

Top 10000 selected

✅ Cardiac_filtered.hg19.rds_selective saved (9974 peaks)


--------------------------------------------

Processing Digestive_filtered.hg19.rds_selective ...

Top 10000 selected

✅ Digestive_filtered.hg19.rds_selective saved (9943 peaks)


--------------------------------------------

Processing Lymphoid_filtered.hg19.rds_selective ...

Top 10000 selected

✅ Lymphoid_filtered.hg19.rds_selective saved (9910 peaks)


--------------------------------------------

Processing Musculoskeletal_filtered.hg19.rds_selective ...

Top 10000 selected

✅ Musculoskeletal_filtered.hg19.rds_selective saved (9892 peaks)


--------------------------------------------

Processing Myeloid_erythro


✅ DHS batch finalization complete. QC summary written.


## Step 2: ATAC file finalizaiton

In [15]:
#!/usr/bin/env Rscript
# ===================================================================
# ATAC Finalization (Top 10 k → collapse ±100 bp + QC)
# ===================================================================

suppressPackageStartupMessages({
  library(GenomicRanges); library(IRanges); library(rtracklayer)
  library(fs); library(tools); library(data.table)
})

indir  <- "../ref/CRE_sites_Selective/scATAC_sites_Selective3"
outdir <- "../ref/CRE_sites_Final/ATAC_filtered_final"
dir_create(outdir)
files  <- list.files(indir, pattern="\\.rds$", full.names=TRUE)
cat(sprintf("Found %d ATAC files\n", length(files)))

qc_all <- list()

for (file in files) {
  tissue <- file_path_sans_ext(basename(file))
  message("\n--------------------------------------------")
  message(sprintf("Processing %s ...", tissue))

  gr <- readRDS(file)
  n_input <- length(gr)

  keep <- !is.na(gr$ATAC_summit_hg19) & !is.na(gr$score)
  gr <- gr[keep]; if(!length(gr)) next

  gr <- gr[order(gr$score, decreasing=TRUE)]
  gr_top <- gr[seq_len(min(10000L, length(gr)))]

  gr_1bp <- GRanges(seqnames=seqnames(gr_top),
                    ranges=IRanges(start=gr_top$ATAC_summit_hg19, width=1),
                    strand="*")
  mcols(gr_1bp) <- mcols(gr_top)[, c(
    "name","score","CRE_source","Class","CRE.module"="CRE module",
    "fold.change"="fold-change","log10pvalue","log10qvalue"
  )]

  clusters <- reduce(gr_1bp, min.gapwidth=200, ignore.strand=TRUE)
  hits <- findOverlaps(gr_1bp, clusters, ignore.strand=TRUE)
  idx_by_cluster <- split(queryHits(hits), subjectHits(hits))
  best_idx <- vapply(idx_by_cluster, function(ix) ix[which.max(gr_1bp$score[ix])], integer(1))
  gr_nonred <- gr_1bp[best_idx]
  gr_nonred <- gr_nonred[order(gr_nonred$score, decreasing=TRUE)]

  qc_all[[tissue]] <- data.frame(
    Cell_Type=tissue, Input_Sites=n_input,
    Top10k_Selected=min(10000L,length(gr)),
    Final_Nonredundant=length(gr_nonred),
    Percent_Retained=round(100*length(gr_nonred)/n_input,2),
    Median_Score=median(gr_nonred$score,na.rm=TRUE),
    Mean_Score=mean(gr_nonred$score,na.rm=TRUE)
  )

  out_rds <- file.path(outdir, paste0(tissue,"_ATAC_filtered_final_summit.hg19.rds"))
  out_bed <- sub("\\.rds$",".bed",out_rds)
  saveRDS(gr_nonred,out_rds); export.bed(gr_nonred,out_bed)
  message(sprintf("✅ Saved %s (%d summits)", tissue,length(gr_nonred)))
}

fwrite(rbindlist(qc_all),file.path(outdir,"ATAC_finalization_QC_summary.csv"))
cat("\n✅ ATAC fast finalization complete\n")


Found 20 ATAC files



--------------------------------------------

Processing Adrenal_Cortical_filtered.hg19.rds_selective ...

✅ Saved Adrenal_Cortical_filtered.hg19.rds_selective (10000 summits)


--------------------------------------------

Processing Adult_Stromal_filtered.hg19.rds_selective ...

✅ Saved Adult_Stromal_filtered.hg19.rds_selective (10000 summits)


--------------------------------------------

Processing Cardiomyocyte_filtered.hg19.rds_selective ...

✅ Saved Cardiomyocyte_filtered.hg19.rds_selective (10000 summits)


--------------------------------------------

Processing Endothelial_filtered.hg19.rds_selective ...

✅ Saved Endothelial_filtered.hg19.rds_selective (10000 summits)


--------------------------------------------

Processing Erythroid_filtered.hg19.rds_selective ...

✅ Saved Erythroid_filtered.hg19.rds_selective (10000 summits)


--------------------------------------------

Processing Fetal_Neuronal_Glial_filtered.hg19.rds_selective ...

✅ Saved Fetal_Neuronal_Glial_filte


✅ ATAC fast finalization complete


## Step 3: Overlap CRE Range Finalization (any overlap) – with QC report

In [17]:
#!/usr/bin/env Rscript
# ===================================================================
# Overlap CRE Range Finalization (Top 1 k → collapse any overlap + QC)
# Dependency-free (no igraph)
# ===================================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(IRanges)
  library(rtracklayer)
  library(fs)
  library(tools)
  library(data.table)
})

indir  <- "../ref/CRE_sites_Overlaps/DHS_ATAC_jaccard_overlaps3"
outdir <- "../ref/CRE_sites_Final/CRE_overlaps_filtered_ranges_final"
dir_create(outdir)
files  <- list.files(indir, pattern="\\.rds$", full.names=TRUE)
cat(sprintf("Found %d overlap files\n", length(files)))

# --- collapse any overlapping ranges using reduce(with.revmap=TRUE) ---
collapse_any_overlap <- function(gr, score_col="score") {
  merged <- reduce(gr, with.revmap=TRUE, ignore.strand=TRUE)
  best_idx <- vapply(mcols(merged)$revmap,
                     function(ix) ix[which.max(mcols(gr)[[score_col]][ix])],
                     integer(1))
  gr[best_idx]
}

qc_all <- list()

for (file in files) {
  pair <- file_path_sans_ext(basename(file))
  message("\n--------------------------------------------")
  message(sprintf("Processing %s ...", pair))

  gr <- readRDS(file)
  if (!inherits(gr, "GRanges") || !length(gr)) next

  score_col <- c("DHS_score","ATAC_score","score")[c("DHS_score","ATAC_score","score") %in% names(mcols(gr))][1]
  if (is.na(score_col)) {
    mcols(gr)$score <- width(gr)
    score_col <- "score"
  }

  n_input <- length(gr)
  gr <- gr[order(mcols(gr)[[score_col]], decreasing=TRUE)]
  gr_top <- gr[seq_len(min(1000L, length(gr)))]
  gr_nonred <- collapse_any_overlap(gr_top, score_col)
  gr_nonred <- gr_nonred[order(mcols(gr_nonred)[[score_col]], decreasing=TRUE)]

  qc_all[[pair]] <- data.frame(
    DHS_ATAC_Pair = pair,
    Input_Sites = n_input,
    Top1k_Selected = min(1000L, length(gr)),
    Final_Nonredundant = length(gr_nonred),
    Percent_Retained = round(100 * length(gr_nonred) / n_input, 2),
    Median_Score = median(mcols(gr_nonred)[[score_col]], na.rm=TRUE),
    Mean_Score = mean(mcols(gr_nonred)[[score_col]], na.rm=TRUE)
  )

  out_rds <- file.path(outdir, paste0(pair, "_CRE_overlap_filtered_ranges.hg19.rds"))
  out_bed <- sub("\\.rds$", ".bed", out_rds)
  saveRDS(gr_nonred, out_rds)
  export.bed(gr_nonred, out_bed)
  message(sprintf("✅ Saved %s (%d ranges)", pair, length(gr_nonred)))
}

fwrite(rbindlist(qc_all), file.path(outdir, "Overlap_CRE_ranges_QC_summary.csv"))
cat("\n✅ Overlap CRE range finalization complete (no igraph required)\n")


Found 15 overlap files



--------------------------------------------

Processing Cancer_epithelial-Skin_Mammary_Epithelial.hg19.consensus.overlaps ...

✅ Saved Cancer_epithelial-Skin_Mammary_Epithelial.hg19.consensus.overlaps (1000 ranges)


--------------------------------------------

Processing Cardiac-Cardiomyocyte.hg19.consensus.overlaps ...

✅ Saved Cardiac-Cardiomyocyte.hg19.consensus.overlaps (1000 ranges)


--------------------------------------------

Processing Digestive-GI_Epithelial.hg19.consensus.overlaps ...

✅ Saved Digestive-GI_Epithelial.hg19.consensus.overlaps (1000 ranges)


--------------------------------------------

Processing Lymphoid-Immune.hg19.consensus.overlaps ...

✅ Saved Lymphoid-Immune.hg19.consensus.overlaps (1000 ranges)


--------------------------------------------

Processing Musculoskeletal-Skeletal_Myocyte.hg19.consensus.overlaps ...

✅ Saved Musculoskeletal-Skeletal_Myocyte.hg19.consensus.overlaps (1000 ranges)


--------------------------------------------

Processin


✅ Overlap CRE range finalization complete (no igraph required)


## Step 4: Overlap CRE Summit Finalization (any overlap) – with QC report

In [18]:
#!/usr/bin/env Rscript
# ===================================================================
# Overlap ATAC Summit Finalization (Top 1 k → collapse ±100 bp + QC)
# ===================================================================

suppressPackageStartupMessages({
  library(GenomicRanges); library(IRanges); library(rtracklayer)
  library(fs); library(tools); library(data.table)
})

indir  <- "../ref/CRE_sites_Overlaps/DHS_ATAC_jaccard_overlaps3"
outdir <- "../ref/CRE_sites_Final/CRE_overlaps_filtered_summit_final"
dir_create(outdir)
files  <- list.files(indir, pattern="\\.rds$", full.names=TRUE)
cat(sprintf("Found %d overlap files\n", length(files)))

collapse_100bp <- function(gr, score_col="score") {
  clusters <- reduce(gr, min.gapwidth=200, ignore.strand=TRUE)
  hits <- findOverlaps(gr, clusters, ignore.strand=TRUE)
  idx_by_cluster <- split(queryHits(hits), subjectHits(hits))
  best_idx <- vapply(idx_by_cluster,function(ix)ix[which.max(mcols(gr)[[score_col]][ix])],integer(1))
  gr[best_idx]
}

qc_all <- list()
for (file in files) {
  pair <- file_path_sans_ext(basename(file))
  message("\n--------------------------------------------")
  message(sprintf("Processing %s ...", pair))

  gr <- readRDS(file)
  if(!inherits(gr,"GRanges")||!length(gr)||!"ATAC_summit_hg19"%in%names(mcols(gr))) next
  score_col <- c("ATAC_score","DHS_score","score")[c("ATAC_score","DHS_score","score")%in%names(mcols(gr))][1]
  if(is.na(score_col)){mcols(gr)$score<-width(gr);score_col<-"score"}

  n_input <- length(gr)
  summit <- mcols(gr)$ATAC_summit_hg19
  in_range <- summit >= start(gr) & summit <= end(gr)
  gr <- gr[in_range]; n_inrange <- length(gr)
  if(!n_inrange) next

  gr <- gr[order(mcols(gr)[[score_col]],decreasing=TRUE)]
  gr_top <- gr[seq_len(min(1000L,length(gr)))]

  gr_1bp <- GRanges(seqnames=seqnames(gr_top),
                    ranges=IRanges(start=mcols(gr_top)$ATAC_summit_hg19,width=1),
                    strand="*")
  mcols(gr_1bp) <- mcols(gr_top)
  gr_nonred <- collapse_100bp(gr_1bp,score_col)
  gr_nonred <- gr_nonred[order(mcols(gr_nonred)[[score_col]],decreasing=TRUE)]

  qc_all[[pair]] <- data.frame(
    DHS_ATAC_Pair=pair, Input_Sites=n_input,
    ATAC_Summits_in_Range=n_inrange,
    Top1k_Selected=min(1000L,length(gr)),
    Final_Nonredundant=length(gr_nonred),
    Percent_Retained=round(100*length(gr_nonred)/n_input,2),
    Median_Score=median(mcols(gr_nonred)[[score_col]],na.rm=TRUE),
    Mean_Score=mean(mcols(gr_nonred)[[score_col]],na.rm=TRUE)
  )

  out_rds <- file.path(outdir,paste0(pair,"_CRE_overlap_filtered_summit.hg19.rds"))
  out_bed <- sub("\\.rds$",".bed",out_rds)
  saveRDS(gr_nonred,out_rds); export.bed(gr_nonred,out_bed)
  message(sprintf("✅ Saved %s (%d summits)",pair,length(gr_nonred)))
}

fwrite(rbindlist(qc_all),file.path(outdir,"Overlap_ATAC_summit_QC_summary.csv"))
cat("\n✅ Overlap ATAC summit fast finalization complete\n")


Found 15 overlap files



--------------------------------------------

Processing Cancer_epithelial-Skin_Mammary_Epithelial.hg19.consensus.overlaps ...

✅ Saved Cancer_epithelial-Skin_Mammary_Epithelial.hg19.consensus.overlaps (1000 summits)


--------------------------------------------

Processing Cardiac-Cardiomyocyte.hg19.consensus.overlaps ...

✅ Saved Cardiac-Cardiomyocyte.hg19.consensus.overlaps (1000 summits)


--------------------------------------------

Processing Digestive-GI_Epithelial.hg19.consensus.overlaps ...

✅ Saved Digestive-GI_Epithelial.hg19.consensus.overlaps (1000 summits)


--------------------------------------------

Processing Lymphoid-Immune.hg19.consensus.overlaps ...

✅ Saved Lymphoid-Immune.hg19.consensus.overlaps (1000 summits)


--------------------------------------------

Processing Musculoskeletal-Skeletal_Myocyte.hg19.consensus.overlaps ...

✅ Saved Musculoskeletal-Skeletal_Myocyte.hg19.consensus.overlaps (1000 summits)


--------------------------------------------

Proc


✅ Overlap ATAC summit fast finalization complete


## Step 5: TCGA cancer subtypes

In [7]:
#!/usr/bin/env Rscript
# ===================================================================
# TCGA ATAC Finalization (Top 5k → collapse ±100 bp + QC)
# Combines Selective3 + Organ3 into a unified output folder
# ===================================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(IRanges)
  library(rtracklayer)
  library(fs)
  library(tools)
  library(data.table)
})

# ------------------------------------------------
# Function to process one TCGA directory
# ------------------------------------------------
process_tcga_dir <- function(indir, outdir, qc_all) {
  files <- list.files(indir, pattern="\\.rds$", full.names=TRUE)
  cat(sprintf("\n🧬 Processing directory: %s (%d files)\n", indir, length(files)))

  for (file in files) {
    tissue <- file_path_sans_ext(basename(file))
    message("\n--------------------------------------------")
    message(sprintf("Processing %s ...", tissue))

    gr <- readRDS(file)
    n_input <- length(gr)
    if (!length(gr)) next

    # Check for 'score' metadata column
    if (is.null(gr$score)) {
      warning(sprintf("No score column for %s — skipping.", tissue))
      next
    }

    # Define summit as midpoint of each region
    gr$midpoint <- start(gr) + floor(width(gr) / 2)

    # Keep only valid positions and scores
    keep <- !is.na(gr$midpoint) & !is.na(gr$score)
    gr <- gr[keep]
    if (!length(gr)) next

    # ------------------------------------------------
    # 🔹 Remove sex chromosomes before ranking
    # ------------------------------------------------
    autosomal <- !(seqnames(gr) %in% c("chrX", "chrY", "X", "Y"))
    gr <- gr[autosomal]
    if (!length(gr)) {
      warning(sprintf("No autosomal peaks remain for %s — skipping.", tissue))
      next
    }

    # ------------------------------------------------
    # Sort by score and select top 5000 peaks
    # ------------------------------------------------
    gr <- gr[order(gr$score, decreasing = TRUE)]
    gr_top <- gr[seq_len(min(5000L, length(gr)))]

    # ------------------------------------------------
    # Create 1 bp GRanges around the midpoint
    # ------------------------------------------------
    gr_1bp <- GRanges(seqnames = seqnames(gr_top),
                      ranges = IRanges(start = gr_top$midpoint, width = 1),
                      strand = "*")

    # Copy selected metadata if present
    mcols(gr_1bp) <- mcols(gr_top)[, intersect(
      c("name", "score", "CRE_source", "Class",
        "fold.change", "log10pvalue", "log10qvalue"),
      names(mcols(gr_top)))
    ]

    # ------------------------------------------------
    # Collapse nearby summits (±100 bp)
    # ------------------------------------------------
    clusters <- reduce(gr_1bp, min.gapwidth = 200, ignore.strand = TRUE)
    hits <- findOverlaps(gr_1bp, clusters, ignore.strand = TRUE)
    idx_by_cluster <- split(queryHits(hits), subjectHits(hits))

    # Retain highest-score peak per cluster (robust)
    best_idx <- vapply(idx_by_cluster, function(ix) {
      if (length(ix) == 0) return(NA_integer_)
      scores <- gr_1bp$score[ix]
      if (all(is.na(scores))) return(NA_integer_)
      ix[which.max(scores)]
    }, integer(1))
    best_idx <- na.omit(best_idx)

    if (length(best_idx) == 0) {
      warning(sprintf("No valid clusters found for %s — skipping.", tissue))
      next
    }

    gr_nonred <- gr_1bp[best_idx]
    gr_nonred <- gr_nonred[order(gr_nonred$score, decreasing = TRUE)]

    # ------------------------------------------------
    # 🔹 Simplified output naming
    # ------------------------------------------------
    base_name <- sub("_.*", "", tissue)  # Strip suffix after first underscore
    out_rds <- file.path(outdir, paste0(base_name, "-ATAC-Final.hg19.rds"))
    out_bed <- file.path(outdir, paste0(base_name, "-ATAC-Final.hg19.bed"))
    dir_create(dirname(out_rds))

    # ------------------------------------------------
    # Save outputs
    # ------------------------------------------------
    saveRDS(gr_nonred, out_rds)
    export.bed(gr_nonred, out_bed)
    message(sprintf("✅ Saved %s (%d summits)", basename(out_rds), length(gr_nonred)))

    # ------------------------------------------------
    # QC summary entry
    # ------------------------------------------------
    qc_all[[length(qc_all) + 1]] <- data.frame(
      Source_Dir = basename(indir),
      Tissue = base_name,
      Input_Sites = n_input,
      Autosomal_Only = sum(autosomal),
      Top5000_Selected = min(5000L, length(gr)),
      Final_Nonredundant = length(gr_nonred),
      Percent_Retained = round(100 * length(gr_nonred) / n_input, 2),
      Median_Score = median(gr_nonred$score, na.rm = TRUE),
      Mean_Score = mean(gr_nonred$score, na.rm = TRUE)
    )
  }

  return(qc_all)
}

# ------------------------------------------------
# Unified output directory
# ------------------------------------------------
outdir <- "../ref/CRE_sites_Final/TCGA_filtered_final"
dir_create(outdir)

# ------------------------------------------------
# Process both input directories
# ------------------------------------------------
qc_all <- list()
qc_all <- process_tcga_dir("../ref/CRE_sites_Selective/TCGA_sites_Selective3", outdir, qc_all)
qc_all <- process_tcga_dir("../ref/CRE_sites_Selective/TCGA_Organ_Selective_v3", outdir, qc_all)

# ------------------------------------------------
# Combine and write QC summary
# ------------------------------------------------
qc_df <- rbindlist(qc_all, fill = TRUE)
qc_path <- file.path(outdir, "TCGA_finalization_QC_summary.csv")
fwrite(qc_df, qc_path)

cat(sprintf("\n✅ All TCGA ATAC finalization complete.\nSummary file: %s\n", qc_path))



🧬 Processing directory: ../ref/CRE_sites_Selective/TCGA_sites_Selective3 (24 files)



--------------------------------------------

Processing ACC_ATAC_Peaks_hg19_selective ...

✅ Saved ACC-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing BLCA_ATAC_Peaks_hg19_selective ...

✅ Saved BLCA-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing BRCA_ATAC_Peaks_hg19_selective ...

✅ Saved BRCA-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing CESC_ATAC_Peaks_hg19_selective ...

✅ Saved CESC-ATAC-Final.hg19.rds (260 summits)


--------------------------------------------

Processing CHOL_ATAC_Peaks_hg19_selective ...

✅ Saved CHOL-ATAC-Final.hg19.rds (654 summits)


--------------------------------------------

Processing COAD_ATAC_Peaks_hg19_selective ...

✅ Saved COAD-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing ESCA_ATAC_Peaks_hg19_selective ...

✅ Saved ESCA-ATAC-Final.hg19.rds (4271 summits)


--


🧬 Processing directory: ../ref/CRE_sites_Selective/TCGA_Organ_Selective_v3 (14 files)



--------------------------------------------

Processing Bladder_organ_selective ...

Warning message in process_tcga_dir("../ref/CRE_sites_Selective/TCGA_Organ_Selective_v3", :
“No valid clusters found for Bladder_organ_selective — skipping.”

--------------------------------------------

Processing Brain_organ_selective ...

Warning message in process_tcga_dir("../ref/CRE_sites_Selective/TCGA_Organ_Selective_v3", :
“No valid clusters found for Brain_organ_selective — skipping.”

--------------------------------------------

Processing Breast_organ_selective ...

Warning message in process_tcga_dir("../ref/CRE_sites_Selective/TCGA_Organ_Selective_v3", :
“No valid clusters found for Breast_organ_selective — skipping.”

--------------------------------------------

Processing Endocrine_organ_selective ...

Warning message in process_tcga_dir("../ref/CRE_sites_Selective/TCGA_Organ_Selective_v3", :
“No valid clusters found for Endocrine_organ_selective — skipping.”

---------------------


✅ All TCGA ATAC finalization complete.
Summary file: ../ref/CRE_sites_Final/TCGA_filtered_final/TCGA_finalization_QC_summary.csv


In [10]:
#!/usr/bin/env Rscript
# ===================================================================
# Organ-Selective + Pan-Cancer CRE Finalization (150–500 bp midpoints)
# Applies uniform QC, filtering, and naming conventions
# ===================================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(IRanges)
  library(rtracklayer)
  library(fs)
  library(tools)
  library(data.table)
})

# ------------------------------------------------
# Input / Output
# ------------------------------------------------
organ_dir  <- "../ref/CRE_sites_Selective/TCGA_Organ_Selective_v3"
pancan_dir <- "../ref/CRE_sites_Selective/TCGA_sites_Selective3"
outdir     <- "../ref/CRE_sites_Final/TCGA_filtered_final"
dir_create(outdir)

# Collect organ-selective + pan-cancer .rds files
organ_files  <- list.files(organ_dir, pattern="\\.rds$", full.names=TRUE)
pancan_file  <- file.path(pancan_dir, "TCGA_pan_cancer_shared_peaks_withscore.rds")
files <- c(organ_files, pancan_file[file.exists(pancan_file)])

cat(sprintf("🧬 Found %d consensus files (organ + pan-cancer)\n", length(files)))

qc_all <- list()

# ------------------------------------------------
# Process each file
# ------------------------------------------------
for (f in files) {
  tissue <- file_path_sans_ext(basename(f))
  message("\n--------------------------------------------")
  message(sprintf("Processing %s ...", tissue))

  gr <- readRDS(f)
  n_input <- length(gr)
  if (!inherits(gr, "GRanges") || n_input == 0) {
    warning(sprintf("Skipping %s (invalid GRanges)", tissue))
    next
  }

  # 1️⃣ Filter by width (150–500 bp)
  gr <- gr[width(gr) > 100 & width(gr) < 1000]
  if (!length(gr)) {
    warning(sprintf("No peaks 150–500 bp for %s", tissue))
    next
  }

  # 2️⃣ Remove sex chromosomes
  autosomal <- !(seqnames(gr) %in% c("chrX", "chrY", "X", "Y"))
  gr <- gr[autosomal]
  if (!length(gr)) {
    warning(sprintf("No autosomal peaks remain for %s", tissue))
    next
  }

  # 3️⃣ Compute 1-bp midpoints
  gr$midpoint <- start(gr) + floor(width(gr) / 2)
  gr_1bp <- GRanges(seqnames = seqnames(gr),
                    ranges = IRanges(start = gr$midpoint, width = 1),
                    strand = "*")
  if ("score" %in% names(mcols(gr))) mcols(gr_1bp)$score <- gr$score

  # 4️⃣ Collapse ±100 bp and keep max-score summit per cluster
  clusters <- reduce(gr_1bp, min.gapwidth = 200, ignore.strand = TRUE)
  hits <- findOverlaps(gr_1bp, clusters, ignore.strand = TRUE)
  idx_by_cluster <- split(queryHits(hits), subjectHits(hits))
  best_idx <- vapply(idx_by_cluster, function(ix) {
    if (length(ix) == 0) return(NA_integer_)
    if (!"score" %in% names(mcols(gr_1bp))) return(ix[1])
    sc <- gr_1bp$score[ix]
    if (all(is.na(sc))) return(ix[1])
    ix[which.max(sc)]
  }, integer(1))
  best_idx <- na.omit(best_idx)
  gr_nonred <- gr_1bp[best_idx]

  # 5️⃣ Sort and take top 5000 by score
  if ("score" %in% names(mcols(gr_nonred)))
    gr_nonred <- gr_nonred[order(gr_nonred$score, decreasing = TRUE)]
  gr_nonred <- gr_nonred[seq_len(min(5000L, length(gr_nonred)))]

  # 6️⃣ Determine base name for output
  if (grepl("pan_cancer", tissue, ignore.case = TRUE)) {
    base_name <- "PanCancer"
  } else {
    base_name <- sub("_.*", "", tissue)
  }

  out_rds <- file.path(outdir, paste0(base_name, "-ATAC-Final.hg19.rds"))
  out_bed <- file.path(outdir, paste0(base_name, "-ATAC-Final.hg19.bed"))

  # 7️⃣ Save results
  saveRDS(gr_nonred, out_rds)
  export.bed(gr_nonred, out_bed)
  message(sprintf("✅ Saved %s (%d summits)", basename(out_rds), length(gr_nonred)))

  # 8️⃣ QC entry
  qc_all[[length(qc_all)+1]] <- data.frame(
    Tissue = base_name,
    Input_Sites = n_input,
    Filtered_150_500bp = sum(width(gr) > 150 & width(gr) < 500),
    Autosomal_Only = sum(autosomal),
    Final_Nonredundant = length(gr_nonred),
    Top5000_Selected = min(5000L, length(gr_nonred)),
    Percent_Retained = round(100 * length(gr_nonred) / n_input, 2),
    Median_Score = median(gr_nonred$score, na.rm = TRUE),
    Mean_Score = mean(gr_nonred$score, na.rm = TRUE),
    Median_Width_Input = median(width(gr), na.rm = TRUE)
  )
}

# ------------------------------------------------
# Write QC summary
# ------------------------------------------------
qc_df <- rbindlist(qc_all, fill = TRUE)
qc_path <- file.path(outdir, "Organ_PanCancer_finalization_QC_summary.csv")
fwrite(qc_df, qc_path)

cat(sprintf("\n✅ Organ + Pan-Cancer finalization complete.\nSummary file: %s\n", qc_path))


🧬 Found 15 consensus files (organ + pan-cancer)



--------------------------------------------

Processing Bladder_organ_selective ...

✅ Saved Bladder-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing Brain_organ_selective ...

✅ Saved Brain-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing Breast_organ_selective ...

✅ Saved Breast-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing Endocrine_organ_selective ...

✅ Saved Endocrine-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing Germ_organ_selective ...

✅ Saved Germ-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing GI_organ_selective ...

✅ Saved GI-ATAC-Final.hg19.rds (5000 summits)


--------------------------------------------

Processing Gyn_organ_selective ...

✅ Saved Gyn-ATAC-Final.hg19.rds (4797 summits)


--------------------------------------------

Proces


✅ Organ + Pan-Cancer finalization complete.
Summary file: ../ref/CRE_sites_Final/TCGA_filtered_final/Organ_PanCancer_finalization_QC_summary.csv


### Final TCGA Script

In [ ]:
#!/usr/bin/env Rscript
# ===================================================================
# Master TCGA + Organ + Pan-Cancer CRE Finalization (100–1000 bp)
# Unified QC, filtering, and naming
# ===================================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(IRanges)
  library(rtracklayer)
  library(fs)
  library(tools)
  library(data.table)
})

# ------------------------------------------------
# Core processing function
# ------------------------------------------------
process_cre_dir <- function(indir, outdir, qc_all) {
  files <- list.files(indir, pattern="\\.rds$", full.names=TRUE)
  cat(sprintf("\n🧬 Processing directory: %s (%d files)\n", indir, length(files)))

  for (file in files) {
    tissue <- file_path_sans_ext(basename(file))
    message("\n--------------------------------------------")
    message(sprintf("Processing %s ...", tissue))

    gr <- readRDS(file)
    n_input <- length(gr)
    if (!inherits(gr, "GRanges") || n_input == 0) {
      warning(sprintf("Skipping %s (invalid GRanges)", tissue))
      next
    }

    # 1️⃣ Filter by width (100–1000 bp)
    gr <- gr[width(gr) > 100 & width(gr) < 1000]
    if (!length(gr)) {
      warning(sprintf("No peaks 100–1000 bp for %s", tissue))
      next
    }

    # 2️⃣ Remove sex chromosomes
    autosomal <- !(seqnames(gr) %in% c("chrX", "chrY", "X", "Y"))
    gr <- gr[autosomal]
    if (!length(gr)) {
      warning(sprintf("No autosomal peaks remain for %s", tissue))
      next
    }

    # 3️⃣ Compute 1-bp midpoints
    gr$midpoint <- start(gr) + floor(width(gr) / 2)
    gr_1bp <- GRanges(seqnames = seqnames(gr),
                      ranges = IRanges(start = gr$midpoint, width = 1),
                      strand = "*")
    if ("score" %in% names(mcols(gr))) mcols(gr_1bp)$score <- gr$score

    # 4️⃣ Collapse nearby summits (±100 bp)
    clusters <- reduce(gr_1bp, min.gapwidth = 200, ignore.strand = TRUE)
    hits <- findOverlaps(gr_1bp, clusters, ignore.strand = TRUE)
    idx_by_cluster <- split(queryHits(hits), subjectHits(hits))
    best_idx <- vapply(idx_by_cluster, function(ix) {
      if (length(ix) == 0) return(NA_integer_)
      sc <- mcols(gr_1bp)$score[ix]
      if (all(is.na(sc))) return(ix[1])
      ix[which.max(sc)]
    }, integer(1))
    best_idx <- na.omit(best_idx)
    gr_nonred <- gr_1bp[best_idx]

    # 5️⃣ Sort and take top 5000 by score
    if ("score" %in% names(mcols(gr_nonred)))
      gr_nonred <- gr_nonred[order(gr_nonred$score, decreasing = TRUE)]
    gr_nonred <- gr_nonred[seq_len(min(5000L, length(gr_nonred)))]

    # 6️⃣ Simplified output naming
    if (grepl("pan_cancer", tissue, ignore.case = TRUE) ||
        grepl("pan_organ", tissue,  ignore.case = TRUE)) {
      base_name <- "PanCancer"
    } else {
      base_name <- sub("_.*", "", tissue)
    }

    out_rds <- file.path(outdir, paste0(base_name, "-ATAC-Final.hg19.rds"))
    out_bed <- file.path(outdir, paste0(base_name, "-ATAC-Final.hg19.bed"))
    dir_create(dirname(out_rds))

    # 7️⃣ Save outputs
    saveRDS(gr_nonred, out_rds)
    export.bed(gr_nonred, out_bed)
    message(sprintf("✅ Saved %s (%d summits)", basename(out_rds), length(gr_nonred)))

    # 8️⃣ QC entry
    qc_all[[length(qc_all)+1]] <- data.frame(
      Source_Dir = basename(indir),
      Tissue = base_name,
      Input_Sites = n_input,
      Autosomal_Only = sum(autosomal),
      Filtered_100_1000bp = length(gr),
      Final_Nonredundant = length(gr_nonred),
      Top5000_Selected = min(5000L, length(gr_nonred)),
      Percent_Retained = round(100 * length(gr_nonred) / n_input, 2),
      Median_Score = median(gr_nonred$score, na.rm = TRUE),
      Mean_Score = mean(gr_nonred$score, na.rm = TRUE),
      Median_Width_Input = median(width(gr), na.rm = TRUE)
    )
  }

  return(qc_all)
}

# ------------------------------------------------
# Unified output directory
# ------------------------------------------------
outdir <- "../ref/CRE_sites_Final/TCGA_filtered_final"
dir_create(outdir)

# ------------------------------------------------
# Run on all three inputs (TCGA + Organ + Pan-Cancer)
# ------------------------------------------------
qc_all <- list()
qc_all <- process_cre_dir("../ref/CRE_sites_Selective/TCGA_sites_Selective3", outdir, qc_all)
qc_all <- process_cre_dir("../ref/CRE_sites_Selective/TCGA_Organ_Selective_v3", outdir, qc_all)

# Include the explicit pan-cancer file if present
pancan_file <- "../ref/CRE_sites_Selective/TCGA_sites_Selective3/TCGA_pan_cancer_shared_peaks_withscore.rds"
if (file.exists(pancan_file)) {
  qc_all <- process_cre_dir(dirname(pancan_file), outdir, qc_all)
}

# ------------------------------------------------
# Combine and write QC summary
# ------------------------------------------------
qc_df <- rbindlist(qc_all, fill = TRUE)
qc_path <- file.path(outdir, "TCGA_Master_finalization_QC_summary.csv")
fwrite(qc_df, qc_path)

cat(sprintf("\n✅ All TCGA, Organ, and Pan-Cancer CRE finalization complete.\nSummary file: %s\n", qc_path))


## Rename files

In [22]:
#!/usr/bin/env Rscript
# ===================================================================
# Final CRE file renamer (dry-run by default)
# Target names:
#   <Tissue>_ATAC_final_summit.hg19.rds / .bed
#   <Tissue>_DHS_final_summit.hg19.rds  / .bed
#   <Pair>_overlap_ranges.hg19.rds      / .bed
#   <Pair>_overlap_summit.hg19.rds      / .bed
# ===================================================================

suppressPackageStartupMessages({
  library(fs)
})

# ---- Toggle this to actually rename ----
DRY_RUN <- FALSE   # set to FALSE to perform renames

# ---- Folders to process ----
dirs <- c(
  "../ref/CRE_sites_Final/DHS_filtered_final",
  "../ref/CRE_sites_Final/ATAC_filtered_final",
  "../ref/CRE_sites_Final/CRE_overlaps_filtered_ranges_final",
  "../ref/CRE_sites_Final/CRE_overlaps_filtered_summit_final"
)

# ---- Helpers ----

ext_of <- function(p) {
  # returns ".rds" / ".bed" etc. (with dot)
  paste0(".", path_ext(p))
}

swap_ext <- function(name, new_ext) {
  sub("\\.[^.]+$", new_ext, name)
}

norm_whitespace <- function(x) {
  gsub("__+", "_", gsub("\\s+", "_", x))
}

# Extract the tissue/pair prefix safely from messy names
extract_prefix_for_DHS <- function(fname) {
  # Try most specific first
  x <- sub("_filtered\\.hg19.*$", "", fname)
  if (x == fname) x <- sub("_DHS_.*$", "", fname)
  if (x == fname) x <- sub("\\.hg19.*$", "", fname) # fallback
  norm_whitespace(x)
}

extract_prefix_for_ATAC <- function(fname) {
  x <- sub("_filtered\\.hg19.*$", "", fname)
  if (x == fname) x <- sub("_ATAC_.*$", "", fname)
  if (x == fname) x <- sub("\\.hg19.*$", "", fname)
  norm_whitespace(x)
}

extract_pair_for_overlap_ranges <- function(fname) {
  # Handles "...hg19.consensus.overlaps_CRE_overlap_filtered_ranges.hg19"
  # and "..._CRE_overlap_filtered_ranges.hg19"
  x <- sub("\\.hg19\\.consensus\\.overlaps_.*$", "", fname)
  x <- sub("_CRE_overlap_filtered_ranges\\.hg19.*$", "", x)
  norm_whitespace(x)
}

extract_pair_for_overlap_summit <- function(fname) {
  x <- sub("_CRE_overlap_filtered_summit\\.hg19.*$", "", fname)
  norm_whitespace(x)
}

# Build target name from detected type + prefix/pair + extension
build_target_name <- function(kind, prefix, ext) {
  # ext should be ".rds" or ".bed"
  stopifnot(grepl("^\\.", ext))
  base <- switch(kind,
    "DHS"     = paste0(prefix, "_DHS_final_summit.hg19", ext),
    "ATAC"    = paste0(prefix, "_ATAC_final_summit.hg19", ext),
    "RANGES"  = paste0(prefix, "_overlap_ranges.hg19", ext),
    "SUMMIT"  = paste0(prefix, "_overlap_summit.hg19", ext),
    stop("Unknown kind: ", kind)
  )
  # collapse any accidental doubles
  base <- gsub("__+", "_", base)
  base
}

# Guess file kind based on folder + filename tokens
detect_kind_and_prefix <- function(dirpath, fname) {
  low <- tolower(fname)

  if (grepl("dhs_filtered_final$", dirpath)) {
    pref <- extract_prefix_for_DHS(fname)
    return(list(kind="DHS", prefix=pref))
  }
  if (grepl("atac_filtered_final$", dirpath)) {
    pref <- extract_prefix_for_ATAC(fname)
    return(list(kind="ATAC", prefix=pref))
  }
  if (grepl("cre_overlaps_filtered_ranges_final$", dirpath)) {
    # prefer explicit marker when present
    if (grepl("cre_overlap_filtered_ranges\\.hg19", low) ||
        grepl("consensus\\.overlaps", low)) {
      pref <- extract_pair_for_overlap_ranges(fname)
      return(list(kind="RANGES", prefix=pref))
    }
  }
  if (grepl("cre_overlaps_filtered_summit_final$", dirpath)) {
    if (grepl("cre_overlap_filtered_summit\\.hg19", low)) {
      pref <- extract_pair_for_overlap_summit(fname)
      return(list(kind="SUMMIT", prefix=pref))
    }
  }

  # Fallback heuristics if folder name is atypical
  if (grepl("overlap.*ranges", low) || grepl("consensus\\.overlaps", low)) {
    pref <- extract_pair_for_overlap_ranges(fname)
    return(list(kind="RANGES", prefix=pref))
  }
  if (grepl("overlap.*summit", low)) {
    pref <- extract_pair_for_overlap_summit(fname)
    return(list(kind="SUMMIT", prefix=pref))
  }
  if (grepl("_dhs_", low)) {
    pref <- extract_prefix_for_DHS(fname)
    return(list(kind="DHS", prefix=pref))
  }
  if (grepl("_atac_", low)) {
    pref <- extract_prefix_for_ATAC(fname)
    return(list(kind="ATAC", prefix=pref))
  }

  NULL
}

# ---- Main pass ----

mapping <- data.frame(
  dir = character(), from = character(), to = character(),
  stringsAsFactors = FALSE
)

for (d in dirs) {
  if (!dir_exists(d)) next
  files <- dir_ls(d, recurse = FALSE, type = "file")

  if (!length(files)) next

  for (f in files) {
    fname <- path_file(f)
    kind_prefix <- detect_kind_and_prefix(d, fname)
    if (is.null(kind_prefix)) next

    ext <- ext_of(f)
    # normalize weird ends like ".hg19" (no extension); assume .rds
    if (!ext %in% c(".rds", ".bed")) {
      if (grepl("\\.bed$", fname, ignore.case = TRUE)) {
        ext <- ".bed"
      } else {
        ext <- ".rds"
      }
    }

    target <- build_target_name(kind_prefix$kind, kind_prefix$prefix, ext)

    # Skip if already correct
    if (identical(fname, target)) next

    old_path <- path(d, fname)
    new_path <- path(d, target)

    mapping <- rbind(mapping, data.frame(dir = d, from = fname, to = target, stringsAsFactors = FALSE))

    if (!DRY_RUN) {
      if (file_exists(new_path)) {
        message("⚠️  Target exists, skipping: ", new_path)
      } else {
        file_move(old_path, new_path)
        message("✅ Renamed: ", fname, " -> ", target)
      }
    }
  }
}

cat("\n------ Proposed renames (DRY_RUN =", DRY_RUN, ") ------\n")
if (nrow(mapping)) {
  print(mapping, row.names = FALSE)
} else {
  cat("No files matched for renaming (already clean or patterns didn’t match).\n")
}
cat("-------------------------------------------------------\n")

if (DRY_RUN) {
  cat("\nTo apply changes, set DRY_RUN <- FALSE and re-run.\n")
}


✅ Renamed: Cancer_epithelial_filtered.hg19.rds_selective_DHS_filtered_final_summit.hg19.bed -> Cancer_epithelial_DHS_final_summit.hg19.bed

✅ Renamed: Cancer_epithelial_filtered.hg19.rds_selective_DHS_filtered_final_summit.hg19.rds -> Cancer_epithelial_DHS_final_summit.hg19.rds

✅ Renamed: Cardiac_filtered.hg19.rds_selective_DHS_filtered_final_summit.hg19.bed -> Cardiac_DHS_final_summit.hg19.bed

✅ Renamed: Cardiac_filtered.hg19.rds_selective_DHS_filtered_final_summit.hg19.rds -> Cardiac_DHS_final_summit.hg19.rds

✅ Renamed: Digestive_filtered.hg19.rds_selective_DHS_filtered_final_summit.hg19.bed -> Digestive_DHS_final_summit.hg19.bed

✅ Renamed: Digestive_filtered.hg19.rds_selective_DHS_filtered_final_summit.hg19.rds -> Digestive_DHS_final_summit.hg19.rds

✅ Renamed: Lymphoid_filtered.hg19.rds_selective_DHS_filtered_final_summit.hg19.bed -> Lymphoid_DHS_final_summit.hg19.bed

✅ Renamed: Lymphoid_filtered.hg19.rds_selective_DHS_filtered_final_summit.hg19.rds -> Lymphoid_DHS_final_summi


------ Proposed renames (DRY_RUN = FALSE ) ------
                                                       dir
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
                 ../ref/CRE_sites_Final/DHS_filtered_final
     

In [24]:
#!/usr/bin/env Rscript
# ===============================================================
# Inspect finalized CRE datasets (DHS / ATAC / overlap-range / overlap-summit)
# ===============================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(data.table)
})

# ---- Example file paths (adjust names if needed) ----
dhs_file   <- "../ref/CRE_sites_Final/DHS_filtered_final/Cancer_epithelial_DHS_final_summit.hg19.rds"
atac_file  <- "../ref/CRE_sites_Final/ATAC_filtered_final/Adrenal_Cortical_ATAC_final_summit.hg19.rds"
ovl_range  <- "../ref/CRE_sites_Final/CRE_overlaps_filtered_ranges_final/Vascular_endothelial-Endothelial_overlap_ranges.hg19.rds"
ovl_summit <- "../ref/CRE_sites_Final/CRE_overlaps_filtered_summit_final/Vascular_endothelial-Endothelial_overlap_ranges.hg19.rds"

# ---- Helper function ----
inspect_gr <- function(path, label) {
  if (!file.exists(path)) {
    cat("\n❌", label, "file not found at", path, "\n"); return(invisible())
  }
  gr <- readRDS(path)
  cat("\n============================================================\n")
  cat(label, "→", path, "\n")
  cat("Class:", class(gr), "| Length:", length(gr), "\n")
  if (!length(gr)) return(invisible())
  cat("Metadata columns:", paste(colnames(mcols(gr)), collapse=", "), "\n")
  cat("Median width:", median(width(gr)), "\n")
  if ("score" %in% colnames(mcols(gr))) {
    cat("Score summary:\n"); print(summary(mcols(gr)$score))
  }
  cat("\nPreview (first 5 rows):\n")
  df <- as.data.frame(gr)
  showcols <- intersect(c("seqnames","start","end", head(colnames(mcols(gr)),4)), colnames(df))
  print(df[1:min(5,nrow(df)), showcols, drop=FALSE])
  invisible(gr)
}

# ---- Inspect each final file type ----
inspect_gr(dhs_file,   "DHS final")
inspect_gr(atac_file,  "ATAC final")
inspect_gr(ovl_range,  "Overlap ranges")
inspect_gr(ovl_summit, "Overlap summits")

cat("\n✅ Inspection complete.\n")



DHS final → ../ref/CRE_sites_Final/DHS_filtered_final/Cancer_epithelial_DHS_final_summit.hg19.rds 
Class: GRanges | Length: 9992 
Metadata columns: name, mean_signal, peak.count, component, score, CRE_source, core_start_hg19, core_end_hg19, summit_hg19, summit_within_core 
Median width: 1 
Score summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.06501 0.08157 0.10835 0.13546 0.15953 1.03392 

Preview (first 5 rows):
  seqnames     start       end      name mean_signal peak.count
1     chr3 159547579 159547579  3.825472    3.399538        246
2     chr7 132542010 132542010  7.850313   99.722600          1
3    chr10 118739721 118739721 10.886676   93.578200          1
4    chr13  42797986  42797986 13.432297   84.846500          1
5     chr3  42518442  42518442  3.292798    2.160184        194
          component
1 Cancer_epithelial
2 Cancer_epithelial
3 Cancer_epithelial
4 Cancer_epithelial
5 Cancer_epithelial

ATAC final → ../ref/CRE_sites_Final/ATAC_filtered_final/Adrenal_C

### TCGA QC

In [11]:
#!/usr/bin/env Rscript
# ===============================================================
# Inspect finalized CRE datasets (TCGA / DHS / ATAC / Overlaps)
# Ensures consistent GRanges formatting for downstream analysis
# ===============================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(IRanges)
  library(data.table)
  library(fs)
})

# ---------------------------------------------------------------
# Define directories for all finalized CRE sets
# ---------------------------------------------------------------
final_dirs <- c(
  "../ref/CRE_sites_Final/DHS_filtered_final",
  "../ref/CRE_sites_Final/ATAC_filtered_final",
  "../ref/CRE_sites_Final/CRE_overlaps_filtered_ranges_final",
  "../ref/CRE_sites_Final/CRE_overlaps_filtered_summit_final",
  "../ref/CRE_sites_Final/TCGA_filtered_final"
)

cat("🔍 Scanning the following final directories:\n")
print(final_dirs)

# ---------------------------------------------------------------
# Helper: Inspect one GRanges file
# ---------------------------------------------------------------
inspect_gr <- function(path) {
  gr <- readRDS(path)
  cat("\n============================================================\n")
  cat("File:", path, "\n")
  cat("Class:", class(gr), "| Length:", length(gr), "\n")

  if (!inherits(gr, "GRanges")) {
    cat("❌ Not a GRanges object\n")
    return(NULL)
  }
  if (!length(gr)) {
    cat("⚠️ Empty GRanges (no ranges)\n")
    return(NULL)
  }

  # --- Basic QC ---
  meta_cols <- colnames(mcols(gr))
  has_score <- "score" %in% meta_cols
  width_vals <- width(gr)
  med_width <- median(width_vals, na.rm = TRUE)
  na_widths <- sum(is.na(width_vals))

  cat("Metadata columns:", paste(meta_cols, collapse=", "), "\n")
  cat("Median width:", med_width, "| NAs:", na_widths, "\n")

  if (has_score) {
    sc <- mcols(gr)$score
    cat("Score summary:\n")
    print(summary(sc))
    if (any(is.na(sc))) cat("⚠️ Contains", sum(is.na(sc)), "NA scores\n")
  } else {
    cat("⚠️ No 'score' column present\n")
  }

  # --- Consistency checks ---
  seqs <- as.character(seqnames(gr))
  if (any(grepl("^chr", seqs)) && any(!grepl("^chr", seqs))) {
    cat("⚠️ Mixed chromosome naming (chr / no-chr)\n")
  }

  if (any(width_vals <= 0)) {
    cat("⚠️ Non-positive widths detected\n")
  }

  # --- Preview ---
  df <- as.data.frame(gr)
  showcols <- intersect(c("seqnames", "start", "end", head(meta_cols, 4)), colnames(df))
  cat("\nPreview (first 5 rows):\n")
  print(df[1:min(5, nrow(df)), showcols, drop = FALSE])
  invisible(gr)
}

# ---------------------------------------------------------------
# Iterate over all .rds files across all final directories
# ---------------------------------------------------------------
all_rds <- unlist(lapply(final_dirs, function(d) {
  if (!dir_exists(d)) return(character(0))
  list.files(d, pattern = "\\.rds$", full.names = TRUE)
}))

cat(sprintf("\n📦 Found %d finalized CRE files total\n", length(all_rds)))

if (!length(all_rds)) stop("❌ No .rds files found — check directory paths.")

# ---------------------------------------------------------------
# Run QC inspection for each file
# ---------------------------------------------------------------
for (f in all_rds) {
  tryCatch({
    inspect_gr(f)
  }, error = function(e) {
    cat("❌ Error inspecting", f, ":", e$message, "\n")
  })
}

cat("\n✅ Comprehensive QC inspection complete.\n")


🔍 Scanning the following final directories:
[1] "../ref/CRE_sites_Final/DHS_filtered_final"                
[2] "../ref/CRE_sites_Final/ATAC_filtered_final"               
[3] "../ref/CRE_sites_Final/CRE_overlaps_filtered_ranges_final"
[4] "../ref/CRE_sites_Final/CRE_overlaps_filtered_summit_final"
[5] "../ref/CRE_sites_Final/TCGA_filtered_final"               

📦 Found 106 finalized CRE files total

File: ../ref/CRE_sites_Final/DHS_filtered_final/Cancer_epithelial_DHS_final_summit.hg19.rds 
Class: GRanges | Length: 9992 
Metadata columns: name, mean_signal, peak.count, component, score, CRE_source, core_start_hg19, core_end_hg19, summit_hg19, summit_within_core 
Median width: 1 | NAs: 0 
Score summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.06501 0.08157 0.10835 0.13546 0.15953 1.03392 

Preview (first 5 rows):
  seqnames     start       end      name mean_signal peak.count
1     chr3 159547579 159547579  3.825472    3.399538        246
2     chr7 132542010 132542010  7.85

In [12]:
#!/usr/bin/env Rscript
# ===============================================================
# Final QC check for all finalized CRE datasets
# Prints ONLY files that pass all QC criteria
# ===============================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(IRanges)
  library(fs)
})

# ---------------------------------------------------------------
# Directories to check
# ---------------------------------------------------------------
final_dirs <- c(
  "../ref/CRE_sites_Final/DHS_filtered_final",
  "../ref/CRE_sites_Final/ATAC_filtered_final",
  "../ref/CRE_sites_Final/CRE_overlaps_filtered_ranges_final",
  "../ref/CRE_sites_Final/CRE_overlaps_filtered_summit_final",
  "../ref/CRE_sites_Final/TCGA_filtered_final"
)

# ---------------------------------------------------------------
# Helper: QC check for one GRanges file
# ---------------------------------------------------------------
qc_pass <- function(path) {
  if (!file.exists(path)) return(FALSE)
  gr <- tryCatch(readRDS(path), error = function(e) NULL)
  if (is.null(gr) || !inherits(gr, "GRanges") || length(gr) == 0) return(FALSE)
  if (!"score" %in% colnames(mcols(gr))) return(FALSE)

  # Width and score checks
  w <- width(gr)
  if (any(is.na(w)) || any(w <= 0) || any(w > 1e4)) return(FALSE)
  sc <- mcols(gr)$score
  if (any(is.na(sc))) return(FALSE)

  # Chromosome naming consistency
  seqs <- as.character(seqnames(gr))
  has_chr <- any(grepl("^chr", seqs))
  no_chr  <- any(!grepl("^chr", seqs))
  if (has_chr && no_chr) return(FALSE)

  TRUE
}

# ---------------------------------------------------------------
# Collect all .rds files and run QC
# ---------------------------------------------------------------
all_rds <- unlist(lapply(final_dirs, function(d) {
  if (!dir_exists(d)) return(character(0))
  list.files(d, pattern = "\\.rds$", full.names = TRUE)
}))

cat(sprintf("🔍 Running final QC on %d finalized CRE files ...\n", length(all_rds)))

# ---------------------------------------------------------------
# Evaluate and print passing files
# ---------------------------------------------------------------
passes <- vapply(all_rds, qc_pass, logical(1))
passed_files <- all_rds[passes]

if (length(passed_files)) {
  cat("\n✅ The following files PASSED QC:\n")
  cat(paste0("   • ", passed_files, collapse = "\n"), "\n")
  cat("\n🎯 Final QC complete — all listed files are ready for downstream analysis.\n")
} else {
  cat("\n⚠️ No files passed QC (check directory paths or formatting).\n")
}


🔍 Running final QC on 106 finalized CRE files ...

✅ The following files PASSED QC:
   • ../ref/CRE_sites_Final/DHS_filtered_final/Cancer_epithelial_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered_final/Cardiac_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered_final/Digestive_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered_final/Lymphoid_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered_final/Musculoskeletal_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered_final/Myeloid_erythroid_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered_final/Neural_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered_final/Organ_devel_renal_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered_final/Placental_trophoblast_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered_final/Primitive_embryonic_DHS_final_summit.hg19.rds
   • ../ref/CRE_sites_Final/DHS_filtered